In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
import torch
import gc

# Limpar cache GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
else:
    raise RuntimeError("GPU CUDA não encontrada")

model_name = "meta-llama/Llama-3.2-3B-Instruct"

print(f"🔄 Carregando {model_name}...")
print(f"💡 Configuração: FP16 sem quantização (melhor performance)")

# Carregar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    token=True
)

# OPÇÃO 1: SEM QUANTIZAÇÃO (RECOMENDADO PARA 3B COM 12GB VRAM)
# Melhor velocidade, qualidade 100%
print("\n⚡ Carregando modelo em FP16 (sem quantização)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda:0",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    token=True
)


model.eval()

# Warmup
warmup_input = tokenizer("Olá", return_tensors="pt").to("cuda")
with torch.inference_mode():
    _ = model.generate(**warmup_input, max_new_tokens=10)

# Configuração de geração OTIMIZADA para 3B
generation_config = GenerationConfig(
    max_new_tokens=512,
    temperature=0.7,        # Balanceado
    top_p=0.9,
    top_k=40,               # Reduzido de 50 (3B precisa de menos)
    repetition_penalty=1.15, # Aumentado para evitar repetição
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=True
)

vram_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\n✅ Modelo Llama 3.2 3B carregado!")
print(f"💾 VRAM: {vram_gb:.1f}GB")

🔄 Carregando meta-llama/Llama-3.2-3B-Instruct...
💡 Configuração: FP16 sem quantização (melhor performance)


`torch_dtype` is deprecated! Use `dtype` instead!



⚡ Carregando modelo em FP16 (sem quantização)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



✅ Modelo Llama 3.2 3B carregado!
💾 VRAM: 6.0GB


In [6]:
import json
import socket
import requests
from urllib.parse import urlparse

from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_SSE_URL = "http://127.0.0.1:18080/sse"


def _assert_local_port_open(sse_url: str, timeout_sec: float = 2.0):
    parsed = urlparse(sse_url)
    host = parsed.hostname or "127.0.0.1"
    port = parsed.port or 80
    try:
        with socket.create_connection((host, port), timeout=timeout_sec):
            return host, port
    except OSError as exc:
        raise ConnectionError(
            f"Sem conexão TCP com {host}:{port}. "
            "Refaça o port-forward antes de continuar."
        ) from exc


def _check_sse_http_endpoint(sse_url: str, timeout_sec: float = 5.0):
    try:
        response = requests.get(
            sse_url,
            headers={"Accept": "text/event-stream"},
            timeout=timeout_sec,
            stream=True,
        )
        return {
            "status_code": response.status_code,
            "content_type": response.headers.get("content-type", ""),
        }
    except Exception as exc:
        raise ConnectionError(
            f"Falha ao acessar o endpoint SSE em {sse_url}: {type(exc).__name__}: {exc}"
        ) from exc


class MCPClient:
    """Cliente MCP usando SSE/ClientSession, com validação prévia da conexão."""

    def __init__(self, sse_url: str):
        self.sse_url = sse_url

    async def list_tools(self):
        host, port = _assert_local_port_open(self.sse_url)
        http_info = _check_sse_http_endpoint(self.sse_url)

        print(f"TCP OK em {host}:{port}")
        print(f"HTTP SSE status: {http_info['status_code']}")
        print(f"HTTP content-type: {http_info['content_type']}")

        async with sse_client(
            self.sse_url,
            timeout=5,
            sse_read_timeout=20,
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools_response = await session.list_tools()

                return [
                    {
                        "name": t.name,
                        "description": getattr(t, "description", "") or "",
                        "input_schema": getattr(t, "inputSchema", None)
                        or getattr(t, "input_schema", None)
                        or {},
                    }
                    for t in tools_response.tools
                ]

    async def call_tool(self, name: str, arguments: dict | None = None):
        _assert_local_port_open(self.sse_url)
        _check_sse_http_endpoint(self.sse_url)

        async with sse_client(
            self.sse_url,
            timeout=5,
            sse_read_timeout=20,
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                result = await session.call_tool(name, arguments or {})
                payload = result.model_dump() if hasattr(result, "model_dump") else result
                return json.dumps(payload, ensure_ascii=False)

In [7]:
import json
from langchain_core.tools import Tool


async def mcp_to_langchain_tools(mcp_client):
    tools = []
    mcp_tools = await mcp_client.list_tools()

    for t in mcp_tools:
        name = t["name"]
        description = t.get("description", "")

        def make_sync_func(tool_name):
            def func(input_str: str = ""):
                raise RuntimeError(
                    f"A tool '{tool_name}' precisa ser usada em contexto async. "
                    "Use agent.ainvoke(...), não agent.invoke(...)."
                )
            return func

        def make_async_func(tool_name):
            async def func(input_str: str = ""):
                args = json.loads(input_str) if input_str else {}
                return await mcp_client.call_tool(tool_name, args)
            return func

        tools.append(
            Tool(
                name=name,
                func=make_sync_func(name),
                coroutine=make_async_func(name),
                description=description,
            )
        )

    return tools

In [9]:
import asyncio
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import pipeline as hf_pipeline

system_prompt = """Você é um agente de observabilidade com acesso obrigatório a ferramentas.

REGRAS OBRIGATÓRIAS:
1) Para qualquer pergunta sobre métricas, SLO, disponibilidade, erros, latência, throughput, CPU, memória ou saúde de serviços, você DEVE chamar uma ferramenta antes de responder.
2) Você NÃO pode responder com conhecimento geral quando a pergunta exigir dado do ambiente.
3) Se a pergunta for ambígua, faça uma pergunta curta de clarificação; não invente valor.
4) Se a ferramenta falhar, informe o erro e peça para tentar novamente.
5) Resposta final deve ser curta, objetiva e baseada no resultado da ferramenta.

MAPEAMENTO DE INTENÇÃO (use tools):
- latência, p95, p99 -> query_golden_metric com metric_name="Latency P95" (ou "Latency P99")
- erros, taxa de erro -> query_golden_metric com metric_name="Error Rate"
- requisições, tráfego, throughput -> query_golden_metric com metric_name="Request Rate"
- cpu -> query_golden_metric com metric_name="CPU Usage"
- memória, memoria -> query_golden_metric com metric_name="Memory Usage"

FORMATO DE AÇÃO:
- Primeiro: chame a tool adequada.
- Depois: responda com:
  - Métrica consultada
  - Valor retornado
  - Janela temporal (se disponível)
  - Interpretação em 1 frase

NUNCA responda “conceitualmente” quando houver tool disponível.
"""

pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    repetition_penalty=1.15,
    return_full_text=False,
)


class AsyncChatHuggingFace(ChatHuggingFace):
    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs):
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(
            None,
            lambda: self._generate(
                messages,
                stop=stop,
                run_manager=run_manager,
                **kwargs
            )
        )


llm = AsyncChatHuggingFace(
    llm=HuggingFacePipeline(pipeline=pipe)
)

print("LLM LangChain pronto:", llm)

Device set to use cuda:0


LLM LangChain pronto: llm=HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x75978c877940>, model_id='meta-llama/Llama-3.2-3B-Instruct') tokenizer=PreTrainedTokenizerFast(name_or_path='meta-llama/Llama-3.2-3B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=Fals

In [4]:
from langchain.agents import create_agent

mcp = MCPClient("http://127.0.0.1:18080/sse")
tools = await mcp_to_langchain_tools(mcp)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)
print(f"Agent criado com {len(tools)} tools")
print("Ferramentas disponíveis:")
for tool in tools:
    print(f"- {tool.name}") 

TCP OK em 127.0.0.1:18080
HTTP SSE status: 200
HTTP content-type: text/event-stream; charset=utf-8


NameError: name 'llm' is not defined

In [21]:
import json

def _pick_metric_name(pergunta: str) -> str | None:
    q = pergunta.lower()

    if any(k in q for k in ["latencia", "latência", "p95"]):
        return "Latency P95"

    if "p99" in q:
        return "Latency P99"

    if any(k in q for k in ["erro", "erros", "error"]):
        return "Error Rate"

    if any(k in q for k in ["requis", "throughput", "trafego", "tráfego"]):
        return "Request Rate"

    if "cpu" in q:
        return "CPU Usage"

    if any(k in q for k in ["memoria", "memória", "ram"]):
        return "Memory Usage"

    return None


pergunta = "Qual o trafego da aplicação?"
metric_name = _pick_metric_name(pergunta)

if metric_name is None:
    print("Pergunta sem métrica clara. Usando agent normal.")

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": pergunta}]
    })

    print(response)

else:
    print(f"Forçando tool: query_golden_metric(metric_name='{metric_name}')")

    raw = await mcp.call_tool(
        "query_golden_metric",
        {"metric_name": metric_name}
    )

    print("Resultado bruto da tool:")
    print(raw)

    # Resumo final via LLM (opcional)
    resumo = await llm.ainvoke(
    f"""Resuma o resultado abaixo para SRE.
    Regras:
    - Não invente unidades
    - Use exatamente a unidade informada em metric_info.unit
    - Não converta req/s para %
    - Se houver múltiplos serviços, liste cada serviço com seu valor

    Resultado:
    {raw}
    """
    )

    print("\nResumo:")
    print(resumo)

Forçando tool: query_golden_metric(metric_name='Request Rate')


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Resultado bruto da tool:
{"meta": null, "content": [{"type": "text", "text": "{\n  \"status\": \"success\",\n  \"data\": {\n    \"resultType\": \"vector\",\n    \"result\": [\n      {\n        \"metric\": {\n          \"name\": \"user\"\n        },\n        \"value\": [\n          1777723549.83,\n          \"19.968502169421853\"\n        ]\n      },\n      {\n        \"metric\": {\n          \"name\": \"payment\"\n        },\n        \"value\": [\n          1777723549.83,\n          \"0.7330812562697738\"\n        ]\n      },\n      {\n        \"metric\": {\n          \"name\": \"catalogue\"\n        },\n        \"value\": [\n          1777723549.83,\n          \"27.48518935527659\"\n        ]\n      },\n      {\n        \"metric\": {\n          \"name\": \"front-end\"\n        },\n        \"value\": [\n          1777723549.83,\n          \"81.75635148075527\"\n        ]\n      },\n      {\n        \"metric\": {\n          \"name\": \"carts\"\n        },\n        \"value\": [\n        

In [5]:
import sys
import json
import socket
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse
from IPython.display import display, HTML

MCP_SERVER_DIR = "/home/daniel/projetos/microservices-demo/mcp-observability-server"
if MCP_SERVER_DIR not in sys.path:
    sys.path.insert(0, MCP_SERVER_DIR)

from loki_client import LokiClient

LOKI_BASE_URL = "http://127.0.0.1:3100"
loki = LokiClient(base_url=LOKI_BASE_URL)


def display_scrollable(title: str, data, height: int = 300):
    text = json.dumps(data, indent=2, ensure_ascii=False) if not isinstance(data, str) else data
    display(HTML(f"""
    <h4>{title}</h4>
    <div style="
        height: {height}px;
        overflow-y: scroll;
        border: 1px solid #ccc;
        padding: 8px;
        background: #1e1e1e;
        color: #d4d4d4;
        font-family: monospace;
        font-size: 12px;
        white-space: pre-wrap;
        border-radius: 4px;
    ">{text}</div>
    """))


def loki_preflight(base_url: str, timeout_sec: float = 2.0) -> bool:
    parsed = urlparse(base_url)
    host = parsed.hostname or "127.0.0.1"
    port = parsed.port or (443 if parsed.scheme == "https" else 80)

    try:
        with socket.create_connection((host, port), timeout=timeout_sec):
            pass
    except OSError as exc:
        print(f"[ERRO] Sem conexão TCP com Loki em {host}:{port}: {exc}")
        print("Dica: valide o port-forward/compose e tente novamente.")
        return False

    try:
        ready = loki.client.get(f"{loki.base_url}/ready")
        print("Loki /ready:", ready.status_code, ready.text.strip())
    except Exception as exc:
        print(f"[ERRO] Falha HTTP ao consultar {loki.base_url}/ready: {type(exc).__name__}: {exc}")
        return False

    return True


query = '{namespace="sock-shop"} |= `error` !~ `(?i)Handle`'

end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(hours=1)
start_ns = str(int(start_dt.timestamp() * 1e9))
end_ns = str(int(end_dt.timestamp() * 1e9))

print("Query:", query)
print("Janela UTC:", start_dt.isoformat(), "->", end_dt.isoformat())

if loki_preflight(LOKI_BASE_URL):
    result = loki.query_range(
        query=query,
        start=start_ns,
        end=end_ns,
        limit=20,
    )

    if result.get("status") == "error":
        print("[ERRO] query_range falhou:", result.get("error"))
    else:
        summary = result.get("data", {}).get("stats", {}).get("summary", {})
        streams = result.get("data", {}).get("result", [])

        print("Status:", result.get("status"))
        print("Streams:", len(streams))
        print("Entries retornadas:", summary.get("totalEntriesReturned"))
        print("Linhas processadas:", summary.get("totalLinesProcessed"))

        display_scrollable("Resumo bruto da consulta", result, height=220)

        log_lines = []
        for stream in streams:
            labels = stream.get("stream", {})
            label_text = " ".join(f"{k}={v}" for k, v in labels.items())
            for ts, line in stream.get("values", []):
                log_lines.append(f"[{label_text}]\n{line}")

        if log_lines:
            display_scrollable("Linhas encontradas", "\n\n".join(log_lines), height=420)
        else:
            print('Nenhuma linha encontrada nesta janela. Teste ampliar para 6h ou usar |~ "(?i)error".')
else:
    print("Consulta ao Loki não executada por falha de conectividade.")

Query: {namespace="sock-shop"} |= `error` !~ `(?i)Handle`
Janela UTC: 2026-05-06T07:46:15.683089+00:00 -> 2026-05-06T08:46:15.683089+00:00
[ERRO] Sem conexão TCP com Loki em 127.0.0.1:3100: [Errno 111] Connection refused
Dica: valide o port-forward/compose e tente novamente.
Consulta ao Loki não executada por falha de conectividade.


In [10]:
from collections import Counter
import re
import json
import sys
from datetime import datetime, timedelta, timezone

# Torna a célula autônoma caso ainda não exista cliente Loki no kernel.
if "loki" not in globals():
    MCP_SERVER_DIR = "/home/daniel/projetos/microservices-demo/mcp-observability-server"
    if MCP_SERVER_DIR not in sys.path:
        sys.path.insert(0, MCP_SERVER_DIR)
    from loki_client import LokiClient
    loki = LokiClient(base_url="http://127.0.0.1:3100")


def _normalize_error_line(line: str) -> str:
    text = line.strip()

    # Mantem codigos HTTP e numeros relevantes (ex.: 504, 500).
    # Remove variacao de tempo de resposta para evitar fragmentacao do agrupamento.
    text = re.sub(r"\b\d+(?:\.\d+)?\s*ms\b", "<latency>", text, flags=re.IGNORECASE)

    # Normaliza apenas IDs hexadecimais longos para reduzir ruido de cardinalidade.
    text = re.sub(r"\b[0-9a-f]{8,}\b", "<id>", text, flags=re.IGNORECASE)
    return text


window_minutes = 15
# Seu Loki está com max_entries_limit=2000.
limit = 2000
error_query = '{level="error"} |= ""'

end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(minutes=window_minutes)
start_ns = str(int(start_dt.timestamp() * 1e9))
end_ns = str(int(end_dt.timestamp() * 1e9))

print(f"Consultando Loki na janela de {window_minutes} minutos...")
print("Query:", error_query)
print("Limite por consulta:", limit)

result = loki.query_range(
    query=error_query,
    start=start_ns,
    end=end_ns,
    limit=limit,
)

if result.get("status") == "error":
    print("Falha na consulta ao Loki:", result.get("error"))
else:
    streams = result.get("data", {}).get("result", [])
    counter = Counter()

    total_lines = 0
    for stream in streams:
        for _, line in stream.get("values", []):
            total_lines += 1
            counter[_normalize_error_line(line)] += 1

    print(f"Linhas de erro analisadas: {total_lines}")

    top10 = counter.most_common(10)
    if not top10:
        print("Nenhum erro encontrado para os filtros e janela selecionados.")
    else:
        print("\nTop 10 erros mais frequentes (mensagem normalizada):")
        for idx, (msg, qty) in enumerate(top10, start=1):
            print(f"{idx:02d}. {qty:>4}x | {msg}")

        top10_json = [
            {"rank": i + 1, "count": qty, "error": msg}
            for i, (msg, qty) in enumerate(top10)
        ]

        if "display_scrollable" in globals():
            display_scrollable("Top 10 erros mais frequentes", json.dumps(top10_json, ensure_ascii=False, indent=2), height=260)
        else:
            print("\nResumo JSON:")
            print(json.dumps(top10_json, ensure_ascii=False, indent=2))

Consultando Loki na janela de 15 minutos...
Query: {level="error"} |= ""
Limite por consulta: 2000
Linhas de erro analisadas: 67

Top 10 erros mais frequentes (mensagem normalizada):
01.   18x | at org.apache.catalina.valves.ErrorReportValve.invoke(ErrorReportValve.java:79) [tomcat-embed-core-8.5.11.jar!/:8.5.11]
02.    9x | com.mongodb.MongoQueryException: Query failed with error code 352 and error message 'Unsupported OP_QUERY command: find. The client driver may require an upgrade. For more details see https://dochub.mongodb.org/core/legacy-opcode-removal' on server carts-db:27017
03.    2x | POST /cart 500 <latency> - 64
04.    2x | DELETE /cart 500 <latency> - -
05.    2x | User cart deleted with status: 500
06.    1x | {"t":{"$date":"2026-05-06T09:28:12.531+00:00"},"s":"I",  "c":"CONTROL",  "id":23377,   "ctx":"SignalHandler","msg":"Received signal","attr":{"signal":15,"error":"Terminated"}}
07.    1x | {"t":{"$date":"2026-05-06T09:28:12.067+00:00"},"s":"I",  "c":"CONTROL",  "id"

In [7]:
import json



# Consulta direta no Prometheus via MCP (sem depender de nomes do catálogo de Golden Metrics)

DB_PROMQL = {

    "MySQL Query Rate": 'rate(mysql_global_status_queries[5m])',

    "MongoDB Ops Rate": 'sum(rate(mongodb_ss_opLatencies_ops[5m])) by (name, op_type)',

    "MongoDB Avg Op Latency": 'sum(rate(mongodb_ss_opLatencies_latency[5m])) by (name, op_type) / sum(rate(mongodb_ss_opLatencies_ops[5m])) by (name, op_type)',

    "Redis Commands Rate": 'rate(redis_commands_processed_total{job="kubernetes-service-endpoints"}[5m])',

}



if "mcp" not in globals():

    if "MCPClient" in globals():

        mcp = MCPClient("http://127.0.0.1:18080/sse")

    else:

        raise RuntimeError(

            "Cliente MCP não encontrado. Execute antes a célula que define MCPClient/mcp."

        )





def _decode_mcp_result(raw: str):

    """Extrai payload útil do retorno da call_tool do MCP."""

    obj = json.loads(raw)



    # Formato comum: {"structuredContent": {"result": "{...json...}"}}

    structured = obj.get("structuredContent", {}) if isinstance(obj, dict) else {}

    if isinstance(structured, dict) and "result" in structured:

        result_text = structured.get("result")

        if isinstance(result_text, str):

            try:

                return json.loads(result_text)

            except Exception:

                return {"raw_result": result_text}



    # Fallback: tenta extrair content[0].text

    content = obj.get("content", []) if isinstance(obj, dict) else []

    if content and isinstance(content, list):

        first = content[0]

        if isinstance(first, dict) and isinstance(first.get("text"), str):

            try:

                return json.loads(first["text"])

            except Exception:

                return {"raw_text": first["text"]}



    return obj





db_metrics_results = {}



for metric_name, promql in DB_PROMQL.items():

    print(f"\n=== {metric_name} ===")

    raw = await mcp.call_tool("prometheus_instant_query", {"query": promql})



    try:

        payload = _decode_mcp_result(raw)

    except Exception as exc:

        payload = {"error": f"Falha ao processar retorno: {type(exc).__name__}: {exc}", "raw": raw}



    db_metrics_results[metric_name] = {

        "query": promql,

        "result": payload,

    }

    print(json.dumps(db_metrics_results[metric_name], ensure_ascii=False, indent=2))



print("\nConsulta concluída. Resultado consolidado em: db_metrics_results")


=== MySQL Query Rate ===
{
  "query": "rate(mysql_global_status_queries[5m])",
  "result": {
    "status": "success",
    "data": {
      "resultType": "vector",
      "result": [
        {
          "metric": {
            "instance": "10.1.8.195:9104",
            "job": "kubernetes-pods",
            "kubernetes_namespace": "sock-shop",
            "kubernetes_pod_name": "catalogue-db-95f56d7b5-g26d5",
            "name": "catalogue-db",
            "pod_template_hash": "95f56d7b5"
          },
          "value": [
            1778057192.786,
            "38.77313828608823"
          ]
        }
      ]
    }
  }
}

=== MongoDB Ops Rate ===
{
  "query": "sum(rate(mongodb_ss_opLatencies_ops[5m])) by (name, op_type)",
  "result": {
    "status": "success",
    "data": {
      "resultType": "vector",
      "result": [
        {
          "metric": {
            "name": "carts-db",
            "op_type": "writes"
          },
          "value": [
            1778057192.947,
           

In [ ]:
import json

# Reaproveita o resultado da célula de Top erros.
if "top10_json" not in globals():
    raise RuntimeError(
        "Variável top10_json não encontrada. Execute antes a célula de Top erros no Loki."
    )

if "llm" not in globals():
    raise RuntimeError(
        "Variável llm não encontrada. Execute antes a célula de inicialização do modelo."
    )

payload = {
    "analysis_window_minutes": globals().get("window_minutes", None),
    "query": globals().get("error_query", None),
    "total_lines_analyzed": globals().get("total_lines", None),
    "top_errors": top10_json,
}

prompt = f"""
Você é um SRE sênior e DEVE ser estritamente fiel aos dados.

DADOS DISPONÍVEIS (única fonte permitida):
{json.dumps(payload, ensure_ascii=False, indent=2)}

TAREFA:
Avalie o ambiente com base SOMENTE nessas mensagens de erro e contagens.

REGRAS DE FIDELIDADE (obrigatórias):
- Não mencionar serviços, bancos, filas, componentes ou métricas que não aparecem explicitamente nos dados.
- Se não houver evidência suficiente para uma afirmação, escreva "evidência insuficiente".
- Não inventar números; usar apenas contagens presentes em top_errors e total_lines_analyzed.

FORMATO DE RESPOSTA:
1) Diagnóstico curto (2-3 frases)
2) Evidências observadas (bullet list com mensagem + contagem)
3) Hipóteses permitidas (até 3), cada uma marcada como "forte" ou "fraca"
4) Quais consultas devem ser realizadas para avançar a investigação (até 3).
""".strip()

avaliacao = await llm.ainvoke(prompt)

print("=== Avaliação do LLM sobre o ambiente (grounded) ===\n")
if hasattr(avaliacao, "content"):
    print(avaliacao.content)
else:
    print(avaliacao)

SyntaxError: invalid syntax (3729048563.py, line 65)

In [9]:
# Teste pós-redeploy: valida se query_golden_metric reconhece as métricas novas
import json

metric_names = [
    "MySQL Query Rate",
    "MongoDB Ops Rate",
    "MongoDB Avg Op Latency",
    "Redis Commands Rate",
]

for name in metric_names:
    raw = await mcp.call_tool("query_golden_metric", {"metric_name": name})
    payload = json.loads(raw)

    result_text = (
        payload.get("structuredContent", {}).get("result")
        if isinstance(payload, dict)
        else None
    )

    print(f"\n=== {name} ===")
    print(result_text if result_text is not None else json.dumps(payload, ensure_ascii=False))


=== MySQL Query Rate ===
{
  "status": "success",
  "data": {
    "resultType": "vector",
    "result": [
      {
        "metric": {
          "instance": "10.1.8.141:9104",
          "job": "kubernetes-pods",
          "kubernetes_namespace": "sock-shop",
          "kubernetes_pod_name": "catalogue-db-95f56d7b5-g26d5",
          "name": "catalogue-db",
          "pod_template_hash": "95f56d7b5"
        },
        "value": [
          1778017587.084,
          "4.418837845646594"
        ]
      }
    ]
  },
  "metric_info": {
    "name": "MySQL Query Rate",
    "description": "MySQL queries processed per second",
    "unit": "queries/s"
  }
}

=== MongoDB Ops Rate ===
{
  "status": "success",
  "data": {
    "resultType": "vector",
    "result": [
      {
        "metric": {
          "name": "orders-db",
          "op_type": "writes"
        },
        "value": [
          1778017587.228,
          "0"
        ]
      },
      {
        "metric": {
          "name": "carts-db",
   

In [17]:
# Teste MCP Tempo: busca traces sem LLM
import json

if "mcp" not in globals():
    mcp = MCPClient("http://127.0.0.1:18080/sse")

tempo_raw = await mcp.call_tool(
    "tempo_search_traces",
    {
        "query": "{ status = error }",
        "limit": 20
    }
)
print(tempo_raw)
print(tempo_raw)

{"meta": null, "content": [{"type": "text", "text": "{\n  \"traces\": [\n    {\n      \"traceID\": \"b830b1a0e438c4e\",\n      \"rootServiceName\": \"front-end-proxy\",\n      \"rootTraceName\": \"carts\",\n      \"startTimeUnixNano\": \"1778018237237478000\",\n      \"durationMs\": 1997,\n      \"spanSet\": {\n        \"spans\": [\n          {\n            \"spanID\": \"0b830b1a0e438c4e\",\n            \"startTimeUnixNano\": \"1778018237237478000\",\n            \"durationNanos\": \"1997904000\",\n            \"attributes\": [\n              {\n                \"key\": \"status\",\n                \"value\": {\n                  \"stringValue\": \"error\"\n                }\n              }\n            ]\n          }\n        ],\n        \"matched\": 1\n      },\n      \"spanSets\": [\n        {\n          \"spans\": [\n            {\n              \"spanID\": \"0b830b1a0e438c4e\",\n              \"startTimeUnixNano\": \"1778018237237478000\",\n              \"durationNanos\": \"1997

In [ ]:
import pandas as pd
import json
from pathlib import Path

if "llm" not in globals():
    raise RuntimeError(
        "Variável llm não encontrada. Execute antes a célula de inicialização do modelo (célula com AsyncChatHuggingFace)."
    )

# Carregar CSV do experimento
csv_path = Path("data/experiment.csv")
if not csv_path.exists():
    print(f"Arquivo não encontrado: {csv_path}")
else:
    df_full = pd.read_csv(csv_path)

    # Excluir aquecimento — não conta para o experimento
    df = df_full[df_full['load_state'] != 'AQUECIMENTO'].copy() if 'load_state' in df_full.columns else df_full.copy()

    print(f"Experimento carregado: {len(df_full)} linhas totais, {len(df)} após remover AQUECIMENTO")
    print(f"Estados presentes: {df['load_state'].unique().tolist() if 'load_state' in df.columns else 'N/A'}")
    print(f"Colunas: {', '.join(df.columns[:10])}... ({len(df.columns)} total)")

    # Colunas de erro para adicionar nota de escala
    error_cols = [c for c in df.columns if 'taxa_erros' in c]

    def metric_stats(col):
        stats = {
            "mean": float(df[col].mean()),
            "min": float(df[col].min()),
            "max": float(df[col].max()),
            "std": float(df[col].std()),
        }
        if 'taxa_erros' in col or 'disponibilidade' in col:
            stats["scale"] = "0 a 1 (ex: 0.3 = 30%)"
        return stats

    # Preparar resumo dos dados para o LLM
    experiment_summary = {
        "total_rows": len(df),
        "rounds": df['round'].unique().tolist() if 'round' in df else None,
        "load_states": df['load_state'].unique().tolist() if 'load_state' in df else None,
        "note_aquecimento": "Linhas com load_state=AQUECIMENTO foram excluídas deste resumo",
        "timestamp_range": {
            "start": df['timestamp_utc'].min() if 'timestamp_utc' in df else None,
            "end": df['timestamp_utc'].max() if 'timestamp_utc' in df else None
        },
        "locust_exit_codes": df['locust_exit_code'].unique().tolist() if 'locust_exit_code' in df else None,
        "metrics_by_service": {
            col: metric_stats(col)
            for col in df.columns
            if '_front' in col or '_carts' in col or '_catalogue' in col or '_user' in col
        },
        "aggregate_metrics": {
            col: metric_stats(col)
            for col in ['trafego', 'taxa_erros', 'tempo_resposta_p95', 'saturacao_cpu', 'saturacao_memoria']
            if col in df.columns
        }
    }

    prompt = f"""
Você é um especialista em observabilidade e performance de microsserviços. 
Analize os dados do experimento de carga abaixo e forneça uma narrativa detalhada.

DADOS DO EXPERIMENTO:
{json.dumps(experiment_summary, ensure_ascii=False, indent=2)}

INSTRUÇÕES:
1. Descreva brevemente o que foi observado nas métricas coletadas, destacando padrões ou anomalias.
2. Analise o comportamento de cada serviço (front, carts, catalogue, user)
3. Identifique gargalos e degradações de performance
4. Os dados já excluem o aquecimento (AQUECIMENTO); analise apenas a execução real.
5. Para métricas de erro e disponibilidade, interprete os valores na escala 0-1 (ex: 0.3 = 30%).

Formate como uma narrativa clara e executiva (máx 800 palavras).
""".strip()

    print("\n=== Solicitando análise ao modelo ===\n")

    narrativa = await llm.ainvoke(prompt)

    print("=== NARRATIVA DO EXPERIMENTO ===\n")
    if hasattr(narrativa, "content"):
        print(narrativa.content)
    else:
        print(narrativa)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Experimento carregado: 52 linhas
Colunas: timestamp_utc, round, load_state, round_start_utc, warmup_start_utc, ready_utc, round_end_utc, locust_exit_code, trafego, taxa_erros... (48 total)

=== Solicitando análise ao modelo ===

=== NARRATIVA DO EXPERIMENTO ===

**Análise das Métricas de Observação**

Após analisar as métricas fornecidas, podemos identificar alguns padrões e anomalias importantes nos dados coletados durante o experimento de carga.

**Métricas Coletadas**

As métricas apresentam informações sobre o desempenho dos serviços front, carts, catalogue e usuário durante a execução do experimento. Alguns pontos notáveis incluem:

*   **Trajeito**: A média da taxa de tráfego total é de aproximadamente 185,04 requisições por segundo, com um mínimo de 102,45 req/s e um máximo de 267,64 req/s.
*   **Taxa Erro**: A média da taxa de erros é de aproximadamente 0,29%, com um mínimo de 0% e um máximo de 0,30%.
*   **Tempo Resposta P95**: A média do tempo de resposta no 95º percentil é d

In [11]:
experiment_summary

{'total_rows': 52,
 'rounds': [1, 2],
 'load_states': ['AQUECIMENTO', 'EXECUCAO', 'PRONTA'],
 'timestamp_range': {'start': '2026-05-08T08:28:41+00:00',
  'end': '2026-05-08T08:41:16+00:00'},
 'locust_exit_codes': [1],
 'metrics_by_service': {'trafego_front': {'mean': 97.18834255906869,
   'min': 0.5966112481107311,
   'max': 198.6872002019692,
   'std': 56.99031779781473},
  'trafego_carts': {'mean': 0.0, 'min': 0.0, 'max': 0.0, 'std': 0.0},
  'trafego_catalogue': {'mean': 26.994764590141813,
   'min': 0.6562593218653674,
   'max': 60.16872488586876,
   'std': 18.379979396130427},
  'trafego_user': {'mean': 17.309422234651862,
   'min': 0.6759455556579717,
   'max': 51.71992423779556,
   'std': 11.852210433059419},
  'taxa_erros_front': {'mean': 0.2186581017780282,
   'min': 0.0,
   'max': 0.3070593330446081,
   'std': 0.10849494544264193},
  'taxa_erros_carts': {'mean': nan, 'min': nan, 'max': nan, 'std': nan},
  'taxa_erros_catalogue': {'mean': nan, 'min': nan, 'max': nan, 'std': nan

In [ ]:
# Teste: resumir CSV do experimento via LLM
import pandas as pd
import json

csv_path = "data/experiment.csv"
df = pd.read_csv(csv_path)

# Montar resumo compacto para não estourar o contexto do modelo
summary = {
    "total_rows": len(df),
    "columns": list(df.columns),
    "head": df.head(5).to_dict(orient="records"),
    "stats": df.describe(include="all").to_dict(),
}

prompt = f"""Você é um especialista em observabilidade. Analise o CSV do experimento abaixo e forneça uma narrativa resumida.

DADOS:
{json.dumps(summary, ensure_ascii=False, indent=2, default=str)}

Forneça um resumo executivo (máx 300 palavras) com os principais padrões observados.
""".strip()

print("=== Enviando ao modelo ===")
narrativa = await llm.ainvoke(prompt)

print("=== NARRATIVA ===")
if hasattr(narrativa, "content"):
    print(narrativa.content)
else:
    print(narrativa)
